# Question 1 - IT Support Cost Optimization Using Data Analysis

This notebook is a runnable workflow template for the IT support cost optimization case study.

**Important limitation:** the company did not provide the original dataset. Therefore, this notebook creates a synthetic sample dataset with similar business columns. Replace the sample-data generation cell with the real company data when available.

**Main outputs:**
- data quality report
- cost-driver summary
- escalation risk model
- expected ticket-cost model
- routing recommendation table
- CSV files that can be shared with management

In [ ]:
# Import core libraries used for data analysis, modeling, and visualization.
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

random_seed=42
rng=np.random.default_rng(random_seed)

## 1. Create or load the dataset

In the real project, replace the synthetic data block with:

```python
tickets=pd.read_csv('company_it_support_tickets.csv')
```

The synthetic data contains ticket-level fields, client context fields, service-quality fields, and pre-routing features.

In [ ]:
# Version-compatible OneHotEncoder creation keeps the notebook runnable across different Colab versions.
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


# Generate a synthetic ticket-level sample dataset for demonstration.
def generate_it_support_sample(n_rows=12000, missing_rate=0.20, random_seed=42):
    rng=np.random.default_rng(random_seed)

    issue_categories=np.array([
        'Cloud migration','Security incident','Network outage','Email issue',
        'Hardware failure','Access request','Database issue','Backup failure'
    ])
    priorities=np.array(['Low','Medium','High','Critical'])
    client_types=np.array(['Small','Medium','Enterprise'])
    service_channels=np.array(['Portal','Email','Phone','Monitoring Alert'])
    engineer_groups=np.array(['Tier 1','Tier 2','Tier 3','Vendor Specialist'])

    issue_category=rng.choice(issue_categories, size=n_rows, p=[0.13,0.12,0.14,0.16,0.11,0.16,0.10,0.08])
    priority=rng.choice(priorities, size=n_rows, p=[0.30,0.42,0.20,0.08])
    client_type=rng.choice(client_types, size=n_rows, p=[0.45,0.35,0.20])
    service_channel=rng.choice(service_channels, size=n_rows, p=[0.35,0.25,0.20,0.20])
    engineer_skill_group=rng.choice(engineer_groups, size=n_rows, p=[0.42,0.34,0.18,0.06])

    base_complexity={
        'Cloud migration':2.2,'Security incident':2.8,'Network outage':2.4,'Email issue':0.8,
        'Hardware failure':1.7,'Access request':0.6,'Database issue':2.3,'Backup failure':1.9
    }
    priority_factor={'Low':0.8,'Medium':1.0,'High':1.5,'Critical':2.2}
    client_factor={'Small':0.9,'Medium':1.05,'Enterprise':1.25}
    engineer_cost={'Tier 1':28,'Tier 2':48,'Tier 3':82,'Vendor Specialist':120}
    sla_target={'Low':24,'Medium':12,'High':6,'Critical':2}

    complexity=np.array([base_complexity[v] for v in issue_category])
    priority_multiplier=np.array([priority_factor[v] for v in priority])
    client_multiplier=np.array([client_factor[v] for v in client_type])
    current_queue_load=rng.integers(5,80,size=n_rows)
    past_ticket_count_30d=rng.poisson(lam=np.where(client_type=='Enterprise',12,np.where(client_type=='Medium',7,4)))
    historical_client_csat=np.clip(rng.normal(4.25,0.35,size=n_rows),1,5)
    client_account_value=np.where(client_type=='Enterprise',rng.normal(90000,18000,size=n_rows),
                          np.where(client_type=='Medium',rng.normal(38000,9000,size=n_rows),rng.normal(12000,4000,size=n_rows)))

    resolution_time_hours=np.clip(
        rng.gamma(shape=2.0, scale=complexity*priority_multiplier*client_multiplier) + current_queue_load/45,
        0.2, 80
    )
    cost_per_engineer_hour=np.array([engineer_cost[v] for v in engineer_skill_group])
    sla_target_hours=np.array([sla_target[v] for v in priority])

    escalation_logit=(-2.0 + 0.55*complexity + 0.35*priority_multiplier + 0.015*current_queue_load +
                      0.04*past_ticket_count_30d - 0.55*(historical_client_csat-4.0))
    escalation_probability=1/(1+np.exp(-escalation_logit))
    is_escalated=rng.binomial(1, np.clip(escalation_probability,0.02,0.92))
    escalation_count=np.where(is_escalated==1, rng.integers(1,4,size=n_rows), 0)
    reopen_flag=rng.binomial(1, np.clip(0.04 + 0.08*is_escalated + 0.04*(resolution_time_hours>sla_target_hours),0,0.45))

    csat_score=np.clip(4.65 - 0.04*resolution_time_hours - 0.25*is_escalated -
                       0.20*reopen_flag + rng.normal(0,0.35,size=n_rows),1,5)
    ticket_cost=resolution_time_hours*cost_per_engineer_hour*(1+0.18*escalation_count)

    tickets=pd.DataFrame({
        'ticket_id':[f'TKT-{i:06d}' for i in range(1,n_rows+1)],
        'client_id':[f'CL-{v:04d}' for v in rng.integers(1,650,size=n_rows)],
        'issue_category':issue_category,
        'priority':priority,
        'client_type':client_type,
        'service_channel':service_channel,
        'engineer_skill_group':engineer_skill_group,
        'historical_client_csat':historical_client_csat.round(2),
        'past_ticket_count_30d':past_ticket_count_30d,
        'current_queue_load':current_queue_load,
        'client_account_value':client_account_value.round(2),
        'sla_target_hours':sla_target_hours,
        'resolution_time_hours':resolution_time_hours.round(2),
        'cost_per_engineer_hour':cost_per_engineer_hour,
        'ticket_cost':ticket_cost.round(2),
        'escalation_count':escalation_count,
        'is_escalated':is_escalated,
        'reopen_flag':reopen_flag,
        'csat_score':csat_score.round(2)
    })

    # Introduce random missing values to mirror the 20% missing-data constraint.
    missing_columns=[
        'issue_category','priority','client_type','engineer_skill_group','historical_client_csat',
        'past_ticket_count_30d','current_queue_load','client_account_value','csat_score'
    ]
    for column in missing_columns:
        mask=rng.random(n_rows)<missing_rate
        tickets.loc[mask,column]=np.nan

    return tickets


# Set use_sample_data to False after the real file is available.
use_sample_data=True

if use_sample_data:
    tickets=generate_it_support_sample()
else:
    tickets=pd.read_csv('company_it_support_tickets.csv')

tickets.head()

## 2. Data quality report

This section checks missing values, duplicate identifiers, impossible values, and target distribution. These checks should be included before any modeling work.

In [ ]:
# Standardize column names for stable downstream code.
tickets.columns=tickets.columns.str.strip().str.lower().str.replace(' ','_')

# Build a missing-value report for every column.
missing_report=(
    tickets.isna().mean().mul(100).round(2)
    .reset_index(name='missing_percent')
    .rename(columns={'index':'column'})
    .sort_values('missing_percent', ascending=False)
)

duplicate_ticket_count=tickets['ticket_id'].duplicated().sum()
negative_resolution_count=(pd.to_numeric(tickets['resolution_time_hours'], errors='coerce')<0).sum()
negative_cost_count=(pd.to_numeric(tickets['ticket_cost'], errors='coerce')<0).sum()

print('Duplicate ticket ids:', duplicate_ticket_count)
print('Negative resolution-time rows:', negative_resolution_count)
print('Negative cost rows:', negative_cost_count)
print('\nMissing-value report:')
display(missing_report.head(12))

print('\nEscalation class distribution:')
display(tickets['is_escalated'].value_counts(normalize=True).rename('share').mul(100).round(2))

## 3. Feature engineering for EDA

The notebook keeps model preprocessing inside scikit-learn pipelines. For business summaries, a cleaned copy is created so grouped cost and CSAT tables can be calculated easily.

In [ ]:
# Create a copy for business analysis and reporting.
tickets_eda=tickets.copy()

# Fill categorical values with Unknown for group-level reporting.
categorical_report_columns=['issue_category','priority','client_type','service_channel','engineer_skill_group']
for column in categorical_report_columns:
    tickets_eda[column]=tickets_eda[column].fillna('Unknown')

# Fill numeric values with median values for report calculations.
numeric_report_columns=[
    'historical_client_csat','past_ticket_count_30d','current_queue_load',
    'client_account_value','csat_score'
]
for column in numeric_report_columns:
    tickets_eda[column]=tickets_eda[column].fillna(tickets_eda[column].median())

# Create business KPI fields.
tickets_eda['low_csat']=(tickets_eda['csat_score']<4.2).astype(int)
tickets_eda['sla_breach']=(tickets_eda['resolution_time_hours']>tickets_eda['sla_target_hours']).astype(int)
tickets_eda['first_contact_resolution']=((tickets_eda['escalation_count']==0)&(tickets_eda['reopen_flag']==0)).astype(int)

kpi_summary=pd.DataFrame({
    'metric':['total_tickets','total_cost','avg_cost_per_ticket','avg_csat','escalation_rate','sla_breach_rate','first_contact_resolution_rate'],
    'value':[
        len(tickets_eda),
        tickets_eda['ticket_cost'].sum(),
        tickets_eda['ticket_cost'].mean(),
        tickets_eda['csat_score'].mean(),
        tickets_eda['is_escalated'].mean(),
        tickets_eda['sla_breach'].mean(),
        tickets_eda['first_contact_resolution'].mean()
    ]
})

display(kpi_summary)

## 4. Cost-driver analysis

Management should receive ranked business tables, not only model scores. This section identifies where support cost is concentrated.

In [ ]:
# Summarize cost, escalation, SLA, and CSAT by issue category.
cost_by_issue=(
    tickets_eda.groupby('issue_category')
    .agg(
        ticket_count=('ticket_id','count'),
        total_cost=('ticket_cost','sum'),
        avg_cost=('ticket_cost','mean'),
        avg_resolution_time=('resolution_time_hours','mean'),
        escalation_rate=('is_escalated','mean'),
        sla_breach_rate=('sla_breach','mean'),
        avg_csat=('csat_score','mean')
    )
    .sort_values('total_cost', ascending=False)
    .reset_index()
)

# Round numeric columns for readable reporting.
numeric_cols=cost_by_issue.select_dtypes(include='number').columns
cost_by_issue[numeric_cols]=cost_by_issue[numeric_cols].round(3)

display(cost_by_issue.head(10))

In [ ]:
# Plot top cost drivers using matplotlib.
top_cost=cost_by_issue.head(8).sort_values('total_cost')

plt.figure(figsize=(9,5))
plt.barh(top_cost['issue_category'], top_cost['total_cost'])
plt.title('Top Issue Categories by Total Support Cost')
plt.xlabel('Total support cost')
plt.ylabel('Issue category')
plt.tight_layout()
plt.show()

## 5. Escalation prediction model

The escalation model predicts whether a ticket is likely to need higher-tier support. In production, only features available before routing should be used.

In [ ]:
# Define pre-routing features for escalation prediction.
categorical_features=['issue_category','priority','client_type','service_channel','engineer_skill_group']
numeric_features=['historical_client_csat','past_ticket_count_30d','current_queue_load','client_account_value','sla_target_hours']
target='is_escalated'

X=tickets[categorical_features+numeric_features]
y=tickets[target]

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.25,random_state=random_seed,stratify=y
)

categorical_pipeline=Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', make_one_hot_encoder())
])

numeric_pipeline=Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor=ColumnTransformer(transformers=[
    ('cat', categorical_pipeline, categorical_features),
    ('num', numeric_pipeline, numeric_features)
])

escalation_model=Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

escalation_model.fit(X_train,y_train)
test_probability=escalation_model.predict_proba(X_test)[:,1]
test_prediction=(test_probability>=0.50).astype(int)

print('ROC-AUC:', round(roc_auc_score(y_test,test_probability),3))
print('PR-AUC:', round(average_precision_score(y_test,test_probability),3))
print('\nClassification report:')
print(classification_report(y_test,test_prediction,zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test,test_prediction))

## 6. Top escalation factors

Coefficients from Logistic Regression provide a simple explanation of which features increase or decrease escalation risk.

In [ ]:
# Extract feature names from the preprocessing pipeline.
feature_names=escalation_model.named_steps['preprocess'].get_feature_names_out()
coefficients=escalation_model.named_steps['model'].coef_[0]

feature_importance=(
    pd.DataFrame({'feature':feature_names,'coefficient':coefficients})
    .assign(abs_coefficient=lambda df: df['coefficient'].abs())
    .sort_values('abs_coefficient', ascending=False)
)

display(feature_importance.head(15))

## 7. Expected cost model

This model estimates ticket cost using pre-routing features. It supports proactive routing and cost planning.

In [ ]:
# Train a simple expected-cost model.
cost_target='ticket_cost'
X_cost=tickets[categorical_features+numeric_features]
y_cost=tickets[cost_target]

X_cost_train,X_cost_test,y_cost_train,y_cost_test=train_test_split(
    X_cost,y_cost,test_size=0.25,random_state=random_seed
)

cost_model=Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(n_estimators=120, random_state=random_seed, n_jobs=-1, min_samples_leaf=5))
])

cost_model.fit(X_cost_train,y_cost_train)
cost_prediction=cost_model.predict(X_cost_test)

mae=mean_absolute_error(y_cost_test,cost_prediction)
rmse=np.sqrt(mean_squared_error(y_cost_test,cost_prediction))

print('Cost model MAE:', round(mae,2))
print('Cost model RMSE:', round(rmse,2))

## 8. Generate routing recommendations

The final management deliverable is an action table. The table ranks tickets by risk and expected cost.

In [ ]:
# Score all sample tickets.
scored=tickets_eda.copy()
scored_features=tickets[categorical_features+numeric_features]

scored['escalation_probability']=escalation_model.predict_proba(scored_features)[:,1]
scored['predicted_ticket_cost']=cost_model.predict(scored_features)

# Convert model outputs into practical routing actions.
conditions=[
    (scored['escalation_probability']>=0.75) | ((scored['priority']=='Critical') & (scored['sla_target_hours']<=2)),
    (scored['escalation_probability']>=0.50) | (scored['predicted_ticket_cost']>=scored['predicted_ticket_cost'].quantile(0.80)),
    (scored['escalation_probability']<=0.25) & (scored['predicted_ticket_cost']<=scored['predicted_ticket_cost'].quantile(0.40))
]
actions=['Senior engineer / priority queue','Tier 2 review','Tier 1 / automation candidate']
scored['recommended_action']=np.select(conditions, actions, default='Standard Tier 1 handling')

recommendation_columns=[
    'ticket_id','client_id','issue_category','priority','client_type',
    'escalation_probability','predicted_ticket_cost','recommended_action'
]

ticket_recommendations=(
    scored[recommendation_columns]
    .sort_values(['escalation_probability','predicted_ticket_cost'], ascending=False)
)

display(ticket_recommendations.head(20))

## 9. Management-level action summary

This section converts ticket-level recommendations into a summary that managers can use.

In [ ]:
action_summary=(
    scored.groupby('recommended_action')
    .agg(
        ticket_count=('ticket_id','count'),
        avg_escalation_probability=('escalation_probability','mean'),
        total_predicted_cost=('predicted_ticket_cost','sum'),
        avg_predicted_cost=('predicted_ticket_cost','mean'),
        avg_csat=('csat_score','mean')
    )
    .sort_values('total_predicted_cost', ascending=False)
    .reset_index()
)

numeric_cols=action_summary.select_dtypes(include='number').columns
action_summary[numeric_cols]=action_summary[numeric_cols].round(3)

display(action_summary)

## 10. Export deliverables

The generated CSV files can be attached to a report or used in a dashboard.

In [ ]:
# Export report tables for review.
cost_by_issue.to_csv('q1_cost_driver_summary.csv', index=False)
ticket_recommendations.to_csv('q1_ticket_recommendations.csv', index=False)
action_summary.to_csv('q1_action_summary.csv', index=False)

print('Generated files:')
print('- q1_cost_driver_summary.csv')
print('- q1_ticket_recommendations.csv')
print('- q1_action_summary.csv')

## 11. Production notes

For the real company system, use this notebook for development and validation, then move the production pipeline to a scalable architecture.

- Batch historical processing: SQL warehouse or PySpark.
- Streaming ingestion: Kafka or cloud event bus.
- Real-time feature calculation: Spark Structured Streaming or Flink.
- Online feature lookup: Redis or feature store.
- Model serving: FastAPI or cloud model endpoint.
- Dashboard: Tableau, Power BI, Superset, or Looker.

**References:** uploaded `guide-2.pdf`, scikit-learn model evaluation documentation, Apache Spark Structured Streaming guide, and Apache Kafka documentation.